In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/02 22:01:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz

--2026-03-02 22:01:25--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-03T03%3A35%3A44Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-03T02%3A35%3A43Z&ske=2026-03-03T03%3A35%3A44Z&sks=b&skv=2018-11-09&sig=Rspsgyq%2BGXPAhSySivyzNJ1DurbFdESCIK7boyUiljQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MjUxMDQ4NSwibmJmIjoxNzcyNTA2ODg1LCJwYXRoIj

In [5]:
# DO NOT run the following code:
# !gzip -dc fhvhv_tripdata_2021-01.csv.gz
# It will try to display the trip data and crash your computer because of memory
# Instead, use the following to extract the file and only show the first 10 lines:
!gzip -dc fhvhv_tripdata_2021-01.csv.gz | head -n 10

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,
gzip: error writing to output: Broken pipe
gzip: fhvhv_tripdata_2021-01.csv.gz: uncompress failed


In [6]:
!wc -l fhvhv_tripdata_2021-01.csv
#Note: I'm not sure what happened, but initially this command wasn't working.
#I think it just took a litlte while to actually uncompress the .gz file into a CSV
#This shows there are 11.9 million rows in the CSV 

 11908469 fhvhv_tripdata_2021-01.csv


In [7]:
# Use spark to read the CSV file in as your dataframe
df = spark.read \
    .option("header", "true") \
    .csv('fhvhv_tripdata_2021-01.csv')

In [8]:
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [9]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

In [10]:
# We can see above that everything is being read as strings, not timestamps or numbers
# Spark doesn't try to figure out the datatypes. So we're going to use pandas to read it in.
# But it won't be good to have a massive dataset for pandas, so we're going to save a smaller file to use with it.

df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [11]:
# Similar to week1, we want to create a schema 
!head -n 1001 fhvhv_tripdata_2021-01.csv > head.csv 

#head -n 101 will keep only the first 101 rows > save to a file called head.csv

In [12]:
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [13]:
!wc -l head.csv

    1001 head.csv


In [14]:
# I need to re-add pandas to my project:
!uv add pandas

Resolved 105 packages in 21ms
Audited 102 packages in 15ms


In [15]:
import pandas as pd

In [16]:
!uv add numpy

Resolved 105 packages in 3ms
Audited 102 packages in 2ms


In [17]:
import numpy as np

In [18]:
df_pandas = pd.read_csv('head.csv')
#Use the smaller head.csv file to read into a pandas dataframe

In [19]:
# Note: If you get any errors about numpy or pandas, it's because you need to install them and then restart the kernal.
# If you do this, restart the kernal and then run all the lines above again.

In [21]:
df_pandas.dtypes

hvfhs_license_num           str
dispatching_base_num        str
pickup_datetime             str
dropoff_datetime            str
PULocationID              int64
DOLocationID              int64
SR_Flag                 float64
dtype: object

In [24]:
#Use spark to turn the pandas df into a spark df
#Changed output to .show(), .schema() to see different output
spark.createDataFrame(df_pandas)

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: string, dropoff_datetime: string, PULocationID: bigint, DOLocationID: bigint, SR_Flag: double]

In [25]:
spark.createDataFrame(df_pandas).show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|    NaN|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|    NaN|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|    NaN|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|    NaN|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|    NaN|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [26]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('SR_Flag', DoubleType(), True)])

In [27]:
# we want to change "longtype" (takes 8 bytes) to integer (takes 4 bytes)
# Copy above output into a text file in VS Code and do some formatting (the AI helped me)
# 'StructType' is scala code 
# Change the datatypes for datetime into timestamp
# Change datatype for SR_Flag into String to allow it to be nullable (looks mostly null in preview)

In [28]:
from pyspark.sql import types

In [43]:
# Create a variable called schema with a list of the datatypes:

schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)]
)

In [44]:
# Now we can have spark read in the CSV to a df and use this schema
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv('fhvhv_tripdata_2021-01.csv')

In [45]:
# Confirm the schema worked
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [46]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [47]:
df.head(10)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DOLocationID=167, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 23, 56), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 38, 5), PULocationID=233, DOLocationID=142, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime=datetime.datetime(2021, 1, 1, 0, 42, 51), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 45, 50), PULocationID=142, DOLocationID=143, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_dat

In [48]:
#Now we want to save the dataframe as a parquet file. 
# First we break the large file into smaller partitions (24 of them)

df = df.repartition(24)

In [36]:
# If you check the SparkJobs (localhost:4040) you can see nothing ran
# This is because the repartition doesn't do anything until the next command with the df
# The df is still the same as it was for now

In [49]:
df.write.parquet('fhvhv/2021/01/', mode='overwrite')

26/03/02 23:37:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/02 23:37:30 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/02 23:37:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

In [40]:
# Seems like the file may have already been in 6 partitions. 
# You can see the partitions in your Week_6 folder under /fhvhv/2021/01
# If you try to rewrite it won't if there are already files there
# In that case you can use df.write.parquet('fhvhv/2021/01/', mode='overwrite') to overwite the files

In [50]:
df = spark.read.parquet('fhvhv/2021/01/')

In [51]:
df

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [53]:
df.printSchema()
#Parquet files are smaller and know the schema

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [57]:
# You can create your own df with only specific vars
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') 

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [59]:
# you can add a filter
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
  .filter(df.hvfhs_license_num == 'HV0003')

DataFrame[pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int]

In [60]:
# See the filtered df
!head -n 10 head.csv

hvfhs_license_num,dispatching_base_num,pickup_datetime,dropoff_datetime,PULocationID,DOLocationID,SR_Flag
HV0003,B02682,2021-01-01 00:33:44,2021-01-01 00:49:07,230,166,
HV0003,B02682,2021-01-01 00:55:19,2021-01-01 01:18:21,152,167,
HV0003,B02764,2021-01-01 00:23:56,2021-01-01 00:38:05,233,142,
HV0003,B02764,2021-01-01 00:42:51,2021-01-01 00:45:50,142,143,
HV0003,B02764,2021-01-01 00:48:14,2021-01-01 01:08:42,143,78,
HV0005,B02510,2021-01-01 00:06:59,2021-01-01 00:43:01,88,42,
HV0005,B02510,2021-01-01 00:50:00,2021-01-01 01:04:57,42,151,
HV0003,B02764,2021-01-01 00:14:30,2021-01-01 00:50:27,71,226,
HV0003,B02875,2021-01-01 00:22:54,2021-01-01 00:30:20,112,255,


In [61]:
#Import functions that Spark already has
#Use F. and hit tab to search through functions
from pyspark.sql import functions as F

In [62]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|pickup_date|dropoff_date|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+-----------+------------+
|           HV0005|              B02510|2021-01-01 14:07:50|2021-01-01 14:15:19|         163|          48|   NULL| 2021-01-01|  2021-01-01|
|           HV0005|              B02510|2021-01-01 16:09:42|2021-01-01 16:27:28|         158|         237|   NULL| 2021-01-01|  2021-01-01|
|           HV0003|              B02875|2021-01-02 23:52:11|2021-01-03 00:01:01|         231|         249|   NULL| 2021-01-02|  2021-01-03|
|           HV0003|              B02875|2021-01-04 06:11:02|2021-01-04 06:26:01|          83|         157|   NULL| 2021-01-04|  2021-01-04|
|           HV0005| 

In [63]:
# Limit df to only the columns you want. Use the new vars you defined to select.
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-01|  2021-01-01|         163|          48|
| 2021-01-01|  2021-01-01|         158|         237|
| 2021-01-02|  2021-01-03|         231|         249|
| 2021-01-04|  2021-01-04|          83|         157|
| 2021-01-01|  2021-01-01|          72|         188|
| 2021-01-01|  2021-01-01|         231|         162|
| 2021-01-01|  2021-01-01|         148|          68|
| 2021-01-03|  2021-01-03|         169|          94|
| 2021-01-03|  2021-01-03|          17|          54|
| 2021-01-03|  2021-01-03|          79|          14|
| 2021-01-02|  2021-01-02|         222|          39|
| 2021-01-01|  2021-01-01|         159|         168|
| 2021-01-02|  2021-01-02|         228|         228|
| 2021-01-02|  2021-01-02|         159|         167|
| 2021-01-05|  2021-01-05|          78|         265|
| 2021-01-02|  2021-01-02|          90|       

In [65]:
# Define your own function. Example using dispatching_base_num above (e.g., values: B02510)
# If the base_num can be divided by 7, do one thing. If not, do another thing.

def crazy_stuff(base_num):
    num = int(base_num[1:])   # start at the integer, which is at position 1 (the B is position 0)
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [67]:
# Test if your function works:
print(crazy_stuff('B02510'))
crazy_stuff('B02884')

e/9ce


's/b44'

In [68]:
# You can use this for unit tests. 
# Now you can define this as a user-defined function:

crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [69]:
#Now create a dataframe where you use this as the base id:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show() # everything above is delayed (transformation) until we execute show (action)

[Stage 21:>                                                         (0 + 1) / 1]

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-01|  2021-01-01|         163|          48|
|  e/9ce| 2021-01-01|  2021-01-01|         158|         237|
|  e/b3b| 2021-01-02|  2021-01-03|         231|         249|
|  e/b3b| 2021-01-04|  2021-01-04|          83|         157|
|  e/9ce| 2021-01-01|  2021-01-01|          72|         188|
|  e/9ce| 2021-01-01|  2021-01-01|         231|         162|
|  e/b35| 2021-01-01|  2021-01-01|         148|          68|
|  e/b33| 2021-01-03|  2021-01-03|         169|          94|
|  e/b48| 2021-01-03|  2021-01-03|          17|          54|
|  e/b38| 2021-01-03|  2021-01-03|          79|          14|
|  e/9ce| 2021-01-02|  2021-01-02|         222|          39|
|  e/9ce| 2021-01-01|  2021-01-01|         159|         168|
|  s/acd| 2021-01-02|  2021-01-02|         228|         228|
|  s/b44| 2021-01-02|  2